# Engineering Knowledge Copilot — RAG reference notebook

This notebook demonstrates a small end-to-end ingestion and retrieval flow using sample engineering documents, Gemini embeddings, PostgreSQL, and pgvector. Run it from the repository root after configuring `.env` and starting PostgreSQL with `docker compose up -d postgres`.

Use synthetic/sample text here. Do not index confidential company documents without access controls and authorization checks.

In [ ]:
import os
from pathlib import Path

import psycopg
from dotenv import load_dotenv
from google import genai
from pgvector import Vector
from pgvector.psycopg import register_vector

load_dotenv(Path(".env"), override=False)

GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
PG_DSN = os.environ["PG_DSN"]
EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-001")

client = genai.Client(api_key=GEMINI_API_KEY)

## 1. Prepare a small sample corpus

These are illustrative examples, not real organisational runbooks or architecture decisions.

In [ ]:
documents = [
    {
        "document_id": "RUNBOOK-007",
        "title": "Order Service Database Failure",
        "section": "Failure Handling",
        "content": (
            "If PostgreSQL is unavailable, do not report the order as created. "
            "Return a retryable service error and preserve request idempotency. "
            "Retry only transient failures with bounded exponential backoff."
        ),
    },
    {
        "document_id": "ADR-014",
        "title": "Order Persistence",
        "section": "Decision",
        "content": (
            "PostgreSQL is the source of truth for order state. "
            "Caches may improve read performance but must not be treated as "
            "the authoritative record of whether an order was committed."
        ),
    },
    {
        "document_id": "SOP-003",
        "title": "Incident Response",
        "section": "Triage",
        "content": (
            "During a production incident, confirm customer impact, inspect "
            "service and database health metrics, and record the timeline. "
            "Escalate to the service owner when the incident exceeds the team's threshold."
        ),
    },
]

## 2. Initialize the database schema

The schema is also maintained in `sql/schema.sql`. On a new local database, this creates the table and vector indexes.

In [ ]:
schema_sql = Path("sql/schema.sql").read_text(encoding="utf-8")

with psycopg.connect(PG_DSN) as conn:
    conn.execute(schema_sql)

print("Schema is ready.")

## 3. Split, embed, and upsert document chunks

For this small example, each sample document is stored as one chunk. A real ingestion pipeline should chunk longer content carefully and retain provenance/version metadata.

In [ ]:
chunks = []
for doc in documents:
    chunks.append({
        **doc,
        "chunk_id": f'{doc["document_id"]}-001',
    })

for chunk in chunks:
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=chunk["content"],
    )
    if not result.embeddings or not result.embeddings[0].values:
        raise RuntimeError(f'No embedding returned for {chunk["chunk_id"]}')

    embedding = result.embeddings[0].values
    with psycopg.connect(PG_DSN) as conn:
        register_vector(conn)
        conn.execute(
            """
            INSERT INTO document_chunks
                (chunk_id, document_id, title, section, content, embedding)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (chunk_id) DO UPDATE SET
                document_id = EXCLUDED.document_id,
                title = EXCLUDED.title,
                section = EXCLUDED.section,
                content = EXCLUDED.content,
                embedding = EXCLUDED.embedding
            """,
            (
                chunk["chunk_id"],
                chunk["document_id"],
                chunk["title"],
                chunk["section"],
                chunk["content"],
                Vector(embedding),
            ),
        )

print(f"Upserted {len(chunks)} sample chunks.")

## 4. Retrieve context for a question

PostgreSQL/pgvector uses cosine distance with the `<=>` operator. A lower distance is a closer vector match; it is not a calibrated confidence score.

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    query_result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
    )
    if not query_result.embeddings or not query_result.embeddings[0].values:
        raise RuntimeError("No query embedding returned")

    query_vector = Vector(query_result.embeddings[0].values)
    with psycopg.connect(PG_DSN) as conn:
        register_vector(conn)
        rows = conn.execute(
            """
            SELECT document_id, title, section, content,
                   embedding <=> %s AS distance
            FROM document_chunks
            ORDER BY embedding <=> %s
            LIMIT %s
            """,
            (query_vector, query_vector, top_k),
        ).fetchall()

    return [
        {
            "document_id": row[0],
            "title": row[1],
            "section": row[2],
            "content": row[3],
            "distance": float(row[4]),
        }
        for row in rows
    ]

question = "What should the order service do if PostgreSQL is unavailable?"
retrieved = retrieve(question)
for item in retrieved:
    print(f'[{item["document_id"]}] distance={item["distance"]:.4f}')
    print(item["content"])
    print()

## 5. Generate a grounded answer

The production service in `app/rag_service.py` includes a relevance check, JSON output validation, and an abstention path. This notebook keeps generation short so the retrieval results remain easy to inspect.

In [ ]:
context = "\n\n".join(
    f'[{item["document_id"]} | {item["title"]} | {item["section"]}]\n{item["content"]}'
    for item in retrieved
)

response = client.models.generate_content(
    model=os.getenv("GEMINI_GENERATION_MODEL", "gemini-3.5-flash-lite"),
    contents=(
        "Answer the question using only the context below. If it is insufficient, "
        "say that the documents do not provide enough evidence. Cite claims with "
        "[DOCUMENT-ID].\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    ),
)

print(response.text or "The model returned an empty response.")

## Important production gaps

- Calibrate retrieval thresholds using a labeled evaluation set rather than choosing a threshold by intuition.
- Validate that each citation supports the claim, not just that the document ID was retrieved.
- Add document-level authorization, ingestion versioning/deletion, observability, rate limits, and retries before handling private or production documentation.
- Never commit `.env` or API credentials to source control.